# Laboratorio 2 - Números (pseudo)aleatorios y Generación de Variables Aleatorias

Simulación Computacional - Universidad de los Llanos

Santiago Barrera Meza - 160005004

In [1]:
import numpy as np
import pandas as pd
from scipy import stats, integrate
from math import sqrt, exp, pi

N = 100

### 5.1.1. MidSquare

Implementar o usar un generador con el método de MidSquare para generar una secuencia de $n=100$
números pseudoaleatorios usando las semillas 1931, 675248, 4567, 12122 y 7179345.

In [2]:
def midsquare(semilla, n):
    d = len(str(semilla))
    x = semilla
    u = []
    for i in range(n):
        cuadrado = str(x * x).zfill(2 * d)
        x = int(cuadrado[d // 2: d // 2 + d])
        u.append(x / 10**d)
    return u


semillas = [1931, 675248, 4567, 12122, 7179345]
midsq = {}

for s in semillas:
    midsq[s] = midsquare(s, N)
    print(s, np.round(midsq[s][:8], 6), "distintos:", len(set(midsq[s])))

1931 [7.287e-01 1.003e-01 6.000e-03 3.600e-03 1.200e-03 1.000e-04 0.000e+00
 0.000e+00] distintos: 7
675248 [0.959861 0.333139 0.981593 0.524817 0.432883 0.387691 0.304311 0.605184] distintos: 100
4567 [0.8574 0.5134 0.3579 0.8092 0.4804 0.0784 0.6146 0.7733] distintos: 22
12122 [0.46942 0.03551 0.12609 0.58986 0.79348 0.96105 0.36171 0.08341] distintos: 88
7179345 [0.429946 0.853735 0.863109 0.956628 0.13713  0.804719 0.57283  0.134094] distintos: 100


### 5.1.2. Generador congruencial mixto

Implementar el generador congruencial mixto $X_{n+1}=(aX_n+c)\bmod m$ y generar una secuencia de
$n=100$ números pseudoaleatorios con los parámetros:

a. $X_0=127$, $a=115$, $c=0$, $m=128$
b. $X_0=43$, $a=25$, $c=5$, $m=48$
c. $X_0=13$, $a=16807$, $c=0$, $m=2^{31}-1$
d. $X_0=29022024$, $a=25214903917$, $c=11$, $m=2^{48}$
e. $X_0=3$, $a=5$, $c=7$, $m=200$

In [3]:
def glc(x0, a, c, m, n):
    x = x0
    u = []
    for i in range(n):
        x = (a * x + c) % m
        u.append(x / m)
    return u


parametros = {
    'a) m=128':    (127, 115, 0, 128),
    'b) m=48':     (43, 25, 5, 48),
    'c) m=2^31-1': (13, 16807, 0, 2**31 - 1),
    'd) m=2^48':   (29022024, 25214903917, 11, 2**48),
    'e) m=200':    (3, 5, 7, 200),
}

glcs = {}
for nombre, p in parametros.items():
    glcs[nombre] = glc(*p, N)
    print(nombre, np.round(glcs[nombre][:8], 6), "distintos:", len(set(glcs[nombre])))

a) m=128 [0.101562 0.679688 0.164062 0.867188 0.726562 0.554688 0.789062 0.742188] distintos: 32
b) m=48 [0.5      0.604167 0.208333 0.3125   0.916667 0.020833 0.625    0.729167] distintos: 48
c) m=2^31-1 [1.02000e-04 7.09991e-01 8.22869e-01 9.62452e-01 9.25974e-01 8.46469e-01
 6.11580e-01 8.25241e-01] distintos: 100
d) m=2^48 [0.831627 0.526397 0.503802 0.012274 0.911276 0.939512 0.039565 0.241278] distintos: 100
e) m=200 [0.11  0.585 0.96  0.835 0.21  0.085 0.46  0.335] distintos: 9


### 5.1.3. Otros generadores

Implementar al menos dos generadores del artículo [Mancilla2000] y generar una secuencia de $n=100$
números aleatorios con parámetros de ejemplo. Se escogieron el generador Lagged-Fibonacci y el
generador Shift-Register.

In [4]:
def lagged_fibonacci(semilla, n, r=17, s=5, m=2**31):
    tabla = []
    x = semilla
    for i in range(r):                     # los primeros r valores con un congruencial
        x = (1103515245 * x + 12345) % m
        tabla.append(x)

    u = []
    for i in range(n):
        tabla.append((tabla[-r] + tabla[-s]) % m)
        u.append(tabla[-1] / m)
    return u


def shift_register(semilla, n):
    x = semilla
    u = []
    for i in range(n):
        x = x ^ ((x << 13) & 0xFFFFFFFF)
        x = x ^ (x >> 17)
        x = (x ^ ((x << 5) & 0xFFFFFFFF)) & 0xFFFFFFFF
        u.append(x / 2**32)
    return u


u_lagfib = lagged_fibonacci(160005004, N)
u_shift = shift_register(160005004, N)

print("Lagged-Fibonacci", np.round(u_lagfib[:8], 6), "distintos:", len(set(u_lagfib)))
print("Shift-Register  ", np.round(u_shift[:8], 6), "distintos:", len(set(u_shift)))

Lagged-Fibonacci [0.068547 0.105763 0.852686 0.587941 0.493748 0.710296 0.804225 0.152054] distintos: 100
Shift-Register   [0.976165 0.625999 0.391776 0.305811 0.772338 0.373488 0.580779 0.006193] distintos: 100


### 5.1.4. Tamaño de ciclo, pruebas de uniformidad y aleatoriedad

Para cada una de las secuencias generadas en 5.1.1, 5.1.2 y 5.1.3 calcular:

a. Tamaño de ciclo o periodo de la secuencia si lo tiene.
b. Prueba de uniformidad (chi-cuadrado con $k=10$ clases, $\alpha=0.05$, valor crítico 16.919).
c. Prueba de aleatoriedad (test de rachas respecto a la media, $\alpha=0.05$, valor crítico 1.96).

In [5]:
def periodo(siguiente, estado, limite=300000):
    # siguiente(estado) devuelve el estado que sigue; retorna (transitorio, ciclo)
    vistos = {}
    for i in range(limite):
        if estado in vistos:
            return vistos[estado], i - vistos[estado]
        vistos[estado] = i
        estado = siguiente(estado)
    return None, None


def paso_midsquare(d):
    return lambda x: int(str(x * x).zfill(2 * d)[d // 2: d // 2 + d])


def paso_glc(a, c, m):
    return lambda x: (a * x + c) % m


def paso_shift(x):
    x = x ^ ((x << 13) & 0xFFFFFFFF)
    x = x ^ (x >> 17)
    return (x ^ ((x << 5) & 0xFFFFFFFF)) & 0xFFFFFFFF


def paso_lagfib(estado):     # el estado son los ultimos 17 valores de la tabla
    return estado[1:] + ((estado[0] + estado[-5]) % 2**31,)

In [6]:
def prueba_chi2(u, k=10):
    obs = np.histogram(u, bins=k, range=(0, 1))[0]
    esperado = len(u) / k
    return np.sum((obs - esperado)**2 / esperado)


def prueba_rachas(u):
    u = np.array(u)
    signos = u > u.mean()
    n1 = int(signos.sum())
    n2 = len(u) - n1
    if n1 == 0 or n2 == 0:
        return np.nan
    rachas = 1 + int(np.sum(signos[1:] != signos[:-1]))
    n = n1 + n2
    mu = 2 * n1 * n2 / n + 0.5
    sigma = sqrt(2 * n1 * n2 * (2 * n1 * n2 - n) / (n**2 * (n - 1)))
    return (rachas - mu) / sigma

In [7]:
# estado inicial del Lagged-Fibonacci, para poder buscarle el periodo
tabla_ini = []
x = 160005004
for i in range(17):
    x = (1103515245 * x + 12345) % 2**31
    tabla_ini.append(x)

secuencias = []
for s in semillas:
    secuencias.append(("MidSquare %d" % s, midsq[s], periodo(paso_midsquare(len(str(s))), s)))
for nombre, (x0, a, c, m) in parametros.items():
    secuencias.append(("GLC " + nombre, glcs[nombre], periodo(paso_glc(a, c, m), x0)))
secuencias.append(("Lagged-Fibonacci", u_lagfib, periodo(paso_lagfib, tuple(tabla_ini))))
secuencias.append(("Shift-Register", u_shift, periodo(paso_shift, 160005004)))

filas = []
for nombre, u, (trans, ciclo) in secuencias:
    chi2 = prueba_chi2(u)
    z = prueba_rachas(u)
    filas.append([nombre,
                  trans if trans is not None else '-',
                  ciclo if ciclo is not None else '> 300000',
                  round(chi2, 2), 'SI' if chi2 <= 16.919 else 'NO',
                  round(z, 3), 'SI' if abs(z) <= 1.96 else 'NO'])

pd.DataFrame(filas, columns=['Generador', 'Transitorio', 'Ciclo', 'Chi2', 'Uniforme?',
                             'Z rachas', 'Aleatorio?'])

,Generador,Transitorio,Ciclo,Chi2,Uniforme?,Z rachas,Aleatorio?
0,MidSquare 1931,7,1,860.6,NO,-7.117,NO
1,MidSquare 675248,210,1,12.4,SI,1.709,SI
2,MidSquare 4567,19,4,109.4,NO,-0.302,SI
3,MidSquare 12122,88,1,52.2,NO,-3.306,NO
4,MidSquare 7179345,848,20,15.0,SI,-1.048,SI
5,GLC a) m=128,0,32,6.8,SI,-0.905,SI
6,GLC b) m=48,0,48,1.2,SI,-0.905,SI
7,GLC c) m=2^31-1,-,> 300000,12.0,SI,0.541,SI
8,GLC d) m=2^48,-,> 300000,4.8,SI,0.318,SI
9,GLC e) m=200,2,8,22.8,NO,-4.923,NO


### 5.2. Uso de números aleatorios para evaluar integrales

Desarrollar los ejercicios 3 al 9 de la sección 3.2 del capítulo 3 de [Ross99]:

3. $\int_0^1 \exp\{e^x\}\,dx$
4. $\int_0^1 (1-x^2)^{3/2}\,dx$
5. $\int_{-2}^{2} e^{x+x^2}\,dx$
6. $\int_0^{\infty} x(1+x^2)^{-2}\,dx$
7. $\int_{-\infty}^{\infty} e^{-x^2}\,dx$
8. $\int_0^1\int_0^1 e^{(x+y)^2}\,dy\,dx$
9. $\int_0^{\infty}\int_0^{x} e^{-(x+y)}\,dy\,dx$

In [8]:
K = 100000
U = glc(2391, 25214903917, 11, 2**48, 2 * K)
U1 = np.array(U[:K])
U2 = np.array(U[K:])       # segundo flujo, para las integrales dobles

resultados = []

def mc(nombre, g, exacto):
    est = g.mean()
    error = g.std(ddof=1) / sqrt(K)
    resultados.append([nombre, round(est, 5), round(error, 5), round(exacto, 5),
                       round(abs(est - exacto), 5)])
    print(nombre, "->", round(est, 5), " exacto:", round(exacto, 5))

In [9]:
# 3.  int_0^1 exp(e^x) dx
mc("Ej 3", np.exp(np.exp(U1)),
   integrate.quad(lambda x: np.exp(np.exp(x)), 0, 1)[0])

# 4.  int_0^1 (1 - x^2)^(3/2) dx
mc("Ej 4", (1 - U1**2)**1.5,
   integrate.quad(lambda x: (1 - x**2)**1.5, 0, 1)[0])

# 5.  int_-2^2 e^(x + x^2) dx     con x = -2 + 4y, dx = 4 dy
x5 = -2 + 4 * U1
mc("Ej 5", 4 * np.exp(x5 + x5**2),
   integrate.quad(lambda x: np.exp(x + x**2), -2, 2)[0])

Ej 3 -> 6.29411  exacto: 6.31656
Ej 4 -> 0.59106  exacto: 0.58905
Ej 5 -> 91.52547  exacto: 93.16275


In [10]:
# 6.  int_0^inf x (1 + x^2)^-2 dx    con x = 1/u - 1, dx = du/u^2
x6 = 1 / U1 - 1
mc("Ej 6", x6 * (1 + x6**2)**(-2) / U1**2,
   integrate.quad(lambda x: x * (1 + x**2)**(-2), 0, np.inf)[0])

# 7.  int_-inf^inf e^(-x^2) dx     es par, o sea 2 veces la integral de 0 a inf
x7 = 1 / U1 - 1
mc("Ej 7", 2 * np.exp(-x7**2) / U1**2,
   integrate.quad(lambda x: np.exp(-x**2), -np.inf, np.inf)[0])

Ej 6 -> 0.50039  exacto: 0.5
Ej 7 -> 1.76822  exacto: 1.77245


In [11]:
# 8.  int int e^((x+y)^2) en [0,1]x[0,1]    con dos uniformes independientes
mc("Ej 8", np.exp((U1 + U2)**2),
   integrate.dblquad(lambda y, x: np.exp((x + y)**2), 0, 1, 0, 1)[0])

# 9.  int_0^inf int_0^x e^-(x+y) dy dx
# con la sugerencia I(y<x) la integral interna llega hasta infinito y se cambian las dos variables
x9 = 1 / U1 - 1
y9 = 1 / U2 - 1
mc("Ej 9", np.exp(-(x9 + y9)) * (y9 < x9) / (U1**2 * U2**2),
   integrate.dblquad(lambda y, x: np.exp(-(x + y)), 0, np.inf, 0, lambda x: x)[0])

pd.DataFrame(resultados, columns=['Ejercicio', 'Monte Carlo', 'Error estandar', 'Exacto', 'Dif'])

Ej 8 -> 4.9075  exacto: 4.89916
Ej 9 -> 0.5008  exacto: 0.5


,Ejercicio,Monte Carlo,Error estandar,Exacto,Dif
0,Ej 3,6.29411,0.01038,6.31656,0.02246
1,Ej 4,0.59106,0.00105,0.58905,0.00202
2,Ej 5,91.52547,0.76420,93.16275,1.63728
3,Ej 6,0.50039,0.00106,0.50000,0.00039
4,Ej 7,1.76822,0.00447,1.77245,0.00423
5,Ej 8,4.90750,0.01906,4.89916,0.00834
6,Ej 9,0.50080,0.00231,0.50000,0.00080


### 5.3. Generación de variables aleatorias discretas

Con el método de la transformada inversa y el generador congruencial mixto de parámetros
$X_0=2391$, $a=25214903917$, $c=11$, $m=2^{48}$, generar los primeros 100 valores aleatorios de:

a. $p_1=0.20$, $p_2=0.15$, $p_3=0.25$, $p_4=0.40$, donde $p_j=P\{X=j\}$
b. $P\{X=1\}=0.3$, $P\{X=2\}=0.2$, $P\{X=3\}=0.35$, $P\{X=4\}=0.15$
c. Generar $n$ valores con la función de masa $p_1=1/3$, $p_2=2/3$ y determinar la proporción de
   valores iguales a 1, con $n=100$, $n=1000$ y $n=10000$.
d. $X$ toma los valores $1,\dots,10$ con probabilidades 0.06, 0.06, 0.06, 0.06, 0.06, 0.15, 0.13,
   0.14, 0.15 y 0.13. Dar un algoritmo que genere $X$ con el método de composición.

In [12]:
def transformada_inversa(u, probabilidades):
    acumulada = 0
    for j, p in enumerate(probabilidades):
        acumulada += p
        if u < acumulada:
            return j + 1
    return len(probabilidades)


def generar_discreta(probabilidades, n):
    u = glc(2391, 25214903917, 11, 2**48, n)
    return [transformada_inversa(v, probabilidades) for v in u]

In [13]:
p_a = [0.20, 0.15, 0.25, 0.40]
p_b = [0.30, 0.20, 0.35, 0.15]
x_a = generar_discreta(p_a, 100)
x_b = generar_discreta(p_b, 100)

print("a)", x_a)
print()
print("b)", x_b)
print()
pd.DataFrame({'teorica a': p_a, 'simulada a': [x_a.count(j) / 100 for j in range(1, 5)],
              'teorica b': p_b, 'simulada b': [x_b.count(j) / 100 for j in range(1, 5)]},
             index=[1, 2, 3, 4])

a) [2, 4, 4, 4, 4, 3, 4, 4, 3, 4, 2, 1, 4, 4, 1, 1, 1, 1, 2, 3, 4, 3, 1, 3, 2, 4, 3, 4, 1, 3, 4, 2, 4, 2, 1, 4, 4, 1, 4, 4, 4, 1, 2, 1, 4, 3, 2, 3, 4, 2, 4, 3, 3, 3, 4, 4, 4, 2, 4, 2, 1, 4, 3, 3, 3, 3, 4, 1, 4, 4, 1, 3, 1, 4, 4, 3, 2, 1, 4, 4, 2, 1, 3, 1, 4, 3, 3, 3, 4, 3, 3, 1, 1, 3, 2, 4, 3, 4, 1, 3]

b) [1, 3, 3, 3, 3, 2, 3, 4, 2, 3, 2, 1, 3, 3, 1, 1, 1, 1, 1, 2, 3, 2, 1, 2, 1, 4, 2, 4, 1, 3, 3, 2, 4, 1, 1, 3, 4, 1, 3, 3, 4, 1, 1, 1, 4, 2, 2, 2, 3, 1, 4, 2, 2, 2, 3, 4, 4, 1, 3, 1, 1, 3, 3, 2, 2, 2, 3, 1, 4, 4, 1, 2, 1, 4, 4, 3, 1, 1, 3, 3, 1, 1, 2, 1, 4, 2, 2, 3, 4, 3, 3, 1, 1, 3, 1, 3, 2, 4, 1, 3]



,teorica a,simulada a,teorica b,simulada b
1,0.20,0.21,0.30,0.32
2,0.15,0.14,0.20,0.22
3,0.25,0.27,0.35,0.29
4,0.40,0.38,0.15,0.17


In [14]:
# c)  p1 = 1/3, p2 = 2/3
for n in [100, 1000, 10000]:
    x_c = generar_discreta([1/3, 2/3], n)
    print("n =", n, " unos:", x_c.count(1), " proporcion:", round(x_c.count(1) / n, 4))

n = 100  unos: 34  proporcion: 0.34
n = 1000  unos: 328  proporcion: 0.328
n = 10000  unos: 3316  proporcion: 0.3316


In [15]:
# d)  composicion: pj = 0.6 * uniforme(1..10) + 0.4 * p2,  con p2 = (pj - 0.06) / 0.4
p_d = [0.06, 0.06, 0.06, 0.06, 0.06, 0.15, 0.13, 0.14, 0.15, 0.13]
p_2 = [(p - 0.06) / 0.4 for p in p_d]
print("p2 =", np.round(p_2, 4))


def generar_composicion(n):
    u = glc(2391, 25214903917, 11, 2**48, 2 * n)     # dos uniformes por valor
    valores = []
    for i in range(n):
        if u[2 * i] < 0.6:
            valores.append(int(10 * u[2 * i + 1]) + 1)
        else:
            valores.append(transformada_inversa(u[2 * i + 1], p_2))
    return valores


x_d = generar_composicion(100)
x_d_grande = generar_composicion(10000)
print(x_d)

p2 = [0.    0.    0.    0.    0.    0.225 0.175 0.2   0.225 0.175]
[8, 9, 7, 10, 8, 1, 9, 2, 2, 5, 8, 4, 10, 10, 6, 7, 7, 7, 6, 9, 6, 2, 8, 5, 7, 7, 5, 10, 6, 7, 8, 5, 5, 6, 10, 4, 9, 8, 1, 9, 1, 1, 8, 6, 8, 1, 6, 7, 9, 6, 6, 8, 4, 8, 4, 10, 8, 1, 2, 4, 7, 5, 4, 10, 2, 10, 7, 7, 4, 6, 4, 4, 6, 6, 10, 7, 6, 4, 1, 6, 2, 8, 4, 5, 10, 8, 9, 5, 10, 7, 7, 1, 10, 6, 7, 5, 7, 1, 7, 4]


In [16]:
pd.DataFrame({'teorica': p_d,
              'n = 100': [x_d.count(j) / 100 for j in range(1, 11)],
              'n = 10000': [x_d_grande.count(j) / 10000 for j in range(1, 11)]},
             index=range(1, 11))

,teorica,n = 100,n = 10000
1,0.06,0.09,0.0577
2,0.06,0.06,0.0627
3,0.06,0.00,0.0586
4,0.06,0.12,0.0561
5,0.06,0.09,0.0623
6,0.15,0.15,0.1526
7,0.13,0.17,0.1274
8,0.14,0.13,0.1402
9,0.15,0.07,0.1557
10,0.13,0.12,0.1267


### 5.4. Simulación ad hoc con generación de variables aleatorias discretas

Repetir lo realizado en el Laboratorio 1 para las secciones 5.1 y 5.2 (simulación de la fila del
banco con 20 clientes y cálculo de las medidas de desempeño), pero generando los tiempos entre
llegadas con un generador de variables aleatorias de Poisson con $\lambda=10$ y los tiempos de
servicio con un generador de variables aleatorias Binomiales con $n=10$ y $p=0.40$.

In [17]:
def poisson_inversa(u, lam):
    i = 0
    p = exp(-lam)
    F = p
    while u >= F:
        i += 1
        p = p * lam / i
        F += p
    return i


def binomial_inversa(u, n, p):
    i = 0
    prob = (1 - p)**n
    F = prob
    while u >= F:
        prob = prob * (p / (1 - p)) * (n - i) / (i + 1)
        i += 1
        F += prob
    return i

In [18]:
# comprobacion de los dos generadores con 10000 valores
poi = [poisson_inversa(v, 10) for v in glc(2391, 25214903917, 11, 2**48, 10000)]
binom = [binomial_inversa(v, 10, 0.40) for v in glc(160005004, 25214903917, 11, 2**48, 10000)]

print("Poisson(10):      media", round(np.mean(poi), 4), " varianza", round(np.var(poi), 4))
print("Binomial(10,0.4): media", round(np.mean(binom), 4), " varianza", round(np.var(binom), 4))

Poisson(10):      media 10.0173  varianza 9.8798
Binomial(10,0.4): media 3.9986  varianza 2.3594


In [19]:
n_clientes = 20
llegadas = [poisson_inversa(v, 10) for v in glc(2391, 25214903917, 11, 2**48, n_clientes)]
servicios = [binomial_inversa(v, 10, 0.40) for v in glc(160005004, 25214903917, 11, 2**48, n_clientes)]

reloj = 0
fin_anterior = 0
filas = []

for i in range(n_clientes):
    entre = 0 if i == 0 else llegadas[i]      # el primer cliente llega en el minuto 0
    reloj += entre
    inicio = max(reloj, fin_anterior)
    espera = inicio - reloj
    fin = inicio + servicios[i]
    ocio = max(0, reloj - fin_anterior)
    filas.append([i + 1, entre, reloj, servicios[i], inicio, espera, fin, fin - reloj, ocio])
    fin_anterior = fin

tabla = pd.DataFrame(filas, columns=['Cliente', 'T entre llegadas', 'T llegada', 'T servicio',
                                     'Inicia servicio', 'Espera cola', 'Termina servicio',
                                     'En sistema', 'Ocio servidor']).set_index('Cliente')
tabla.loc['TOTAL'] = tabla.sum()
tabla

,T entre llegadas,T llegada,T servicio,Inicia servicio,Espera cola,Termina servicio,En sistema,Ocio servidor
Cliente,,,,,,,,
1,0,0,4,0,0,4,4,0
2,12,12,7,12,0,19,7,8
3,11,23,3,23,0,26,3,4
4,12,35,4,35,0,39,4,9
5,12,47,2,47,0,49,2,8
6,9,56,6,56,0,62,6,7
7,11,67,4,67,0,71,4,5
8,14,81,1,81,0,82,1,10
9,9,90,3,90,0,93,3,8


In [20]:
espera = tabla['Espera cola'].iloc[:-1]
esperaron = espera[espera > 0]
tiempo_total = tabla['Termina servicio'].iloc[n_clientes - 1]

medidas = {
    'Average time in system': tabla['En sistema'].iloc[:-1].mean(),
    'Percent idle time': tabla['Ocio servidor'].iloc[:-1].sum() / tiempo_total,
    'Average waiting time per customer': espera.mean(),
    'Fraction having to wait': len(esperaron) / n_clientes,
    'Average waiting time of those who waited': esperaron.mean(),
}

# medidas de la primera corrida del Laboratorio 1
lab1 = [3.55, 0.398058, 0.45, 0.30, 1.50]

pd.DataFrame({'Lab 1': lab1, 'Lab 2': list(medidas.values())}, index=list(medidas.keys()))

,Lab 1,Lab 2
Average time in system,3.550000,4.000000
Percent idle time,0.398058,0.572973
Average waiting time per customer,0.450000,0.050000
Fraction having to wait,0.300000,0.050000
Average waiting time of those who waited,1.500000,1.000000
